# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.5 | NumPy 2.3.3 | pandas 3.0.5
Wrote 5 data files to /Users/apple/Documents/Codex/UCSF 26AUT/DATASCI217/ds217-26f-06/data
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

**TODO:** Replace this text with an evidence map: one bullet for each of Lectures 01 to 06. For 01, 02, 04, 05, and 06, name one concrete technique or file and say in one sentence where this notebook or its outputs use it. For 03, complete the NumPy calculation after Task 4.3 and explain your axis choice in one sentence here. The lectures cover the command line and Python basics (01); Git, functions, and files (02); environments, shell pipelines, and NumPy (03); Jupyter and labeled pandas data (04); data contracts, cleaning, and validation (05); and joins, reshaping, and concatenation (06). Do not paste earlier solutions.

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [3]:
%%bash
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [4]:
def profile(name, table):
    return {"table": name, "rows": len(table), "columns": len(table.columns), "exact_duplicate_rows": table.duplicated().sum()}

results = []
for name in manifest["files"]:
    results.append(profile(name, raw[Path(name).stem]))

table_profile = pd.DataFrame(results)
table_profile.to_csv(PROFILE_PATH, index=False)
table_profile

,table,rows,columns,exact_duplicate_rows
0,patients.parquet,30045,4,45
1,sites.csv,5,3,0
2,visits_north.csv,26297,6,35
3,visits_south.csv,31074,6,35
4,a1c_wide.parquet,29424,3,0


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

After cleaning, patients.parquet should have one row per registered patient, with patient_id as the key. sites.csv should have one current record per site, with site_code as the key.

After stacking visits_north.csv and visits_south.csv and keeping the later recheck, each row should represent one patient and visit. The composite key is patient_id and visit.

a1c_wide.parquet should have one row per patient, with patient_id as the key. After reshaping to long form, the key is patient_id and visit.

Raw exact duplicates violate the intended keys. Rechecks repeat the same patient and visit, while current and retired site records repeat site_code.

For Task 4.4, visits to patients is many-to-one on patient_id, and patients to sites is many-to-one on site_code. Visits to long-form A1c results is one-to-one on patient_id and visit.

### 2.2 Stack the two visit files

In [5]:
north = raw["visits_north"].copy(deep=True)
south = raw["visits_south"].copy(deep=True).rename(columns={"wt_kg": "weight_kg"})
north["source_file"] = "visits_north.csv"
south["source_file"] = "visits_south.csv"
visits_raw = pd.concat([north, south], ignore_index=True)
visits_raw.shape

(57371, 7)

### 2.3 Audit the raw tables

In [9]:
patients = raw["patients"]
sbp = pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce")
valid_date = visits_raw["visit_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}") & pd.to_datetime(visits_raw["visit_date"], format="%Y-%m-%d", errors="coerce").notna()
visit_map = {"baseline": "baseline", "bl": "baseline", "week12": "week12", "wk12": "week12", "week 12": "week12"}
visit_pairs = visits_raw[["patient_id", "visit"]].copy(deep=True)
visit_pairs["visit"] = visit_pairs["visit"].str.strip().str.lower().map(visit_map)

issue_audit = pd.DataFrame([
    ("patients: gender values that are not a code", ((patients["gender"].str.strip() != "") & ~patients["gender"].isin(["F", "M", "X"])).sum()),
    ("patients: empty gender", (patients["gender"].str.strip() == "").sum()),
    ("patients: birth_year sentinel 9999", (patients["birth_year"] == "9999").sum()),
    ("patients: site codes needing normalization", (patients["site_code"] != patients["site_code"].str.upper().str.replace("-", "").str.replace(" ", "")).sum()),
    ("sites: rows with a repeated site_code", raw["sites"].duplicated(subset=["site_code"], keep=False).sum()),
    ("visits: visit labels that are not baseline or week12", (~visits_raw["visit"].isin(["baseline", "week12"])).sum()),
    ("visits: sbp sentinel 999", (sbp == 999).sum()),
    ("visits: sbp outside 70 through 250", ((sbp != 999) & ((sbp < 70) | (sbp > 250))).sum()),
    ("visits: date parse failures", (~valid_date).sum()),
    ("visits: rows in repeated patient_id and visit pairs", visit_pairs.duplicated(subset=["patient_id", "visit"], keep=False).sum())
], columns=["issue", "count"])
issue_audit.to_csv(AUDIT_PATH, index=False)

assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."

## Task 3: Clean

### 3.1 Clean the patients

In [ ]:
print(raw["patients"]["gender"].value_counts())
patients_clean = raw["patients"].copy(deep=True).drop_duplicates()
gender_map = {"f": "F", "female": "F", "woman": "F", "m": "M", "male": "M", "man": "M", "x": "X", "nonbinary": "X", "non-binary": "X", "nb": "X"}
patients_clean["gender"] = patients_clean["gender"].str.strip().str.lower().map(gender_map).astype("category")
patients_clean["birth_year"] = patients_clean["birth_year"].replace("9999", pd.NA).astype("Int64")
patients_clean["site_code"] = patients_clean["site_code"].str.upper().str.replace("-", "").str.replace(" ", "")
patients_clean = patients_clean.reset_index(drop=True)

gender
F             11248
M             10971
male           1039
female         1018
Male            622
f               622
Female          578
m               538
woman           400
                382
X               372
man             329
unknown         282
MALE            256
FEMALE          231
 M              223
 F              205
Woman           131
nonbinary       118
                104
NB              104
Unknown          96
non-binary       80
x                62
Non-Binary       34
Name: count, dtype: int64[pyarrow]


In [ ]:
print(patients_clean["gender"].cat.categories)

patients_summary = pd.DataFrame({
    "metric": ["patients", "birth_year missing", "birth_year min", "birth_year max", "site_code values", "patients at N01", "patients at S02", "patients at E04"],
    "value": [len(patients_clean), patients_clean["birth_year"].isna().sum(), patients_clean["birth_year"].min(), patients_clean["birth_year"].max(), patients_clean["site_code"].nunique(), (patients_clean["site_code"] == "N01").sum(), (patients_clean["site_code"] == "S02").sum(), (patients_clean["site_code"] == "E04").sum()]
})
patients_summary.to_csv(PATIENTS_SUMMARY_PATH, index=False)

gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)
print(patients_summary)
gender_counts

Index(['F', 'M', 'X'], dtype='str')
               metric  value
0            patients  30000
1  birth_year missing    371
2      birth_year min   1941
3      birth_year max   1996
4    site_code values      3
5     patients at N01  13739
6     patients at S02  15706
7     patients at E04    555


,gender,count,code
0,F,14413,0
1,M,13955,1
2,NaN,863,-1
3,X,769,2


### 3.2 Clean the visits

In [17]:
print(visits_raw["visit"].value_counts())
visits_clean = visits_raw.copy(deep=True).drop_duplicates()
visits_clean["visit"] = visits_clean["visit"].str.strip().str.lower().map(visit_map)
visits_clean["sbp_mmhg"] = pd.to_numeric(visits_clean["sbp_mmhg"], errors="coerce")
visits_clean["sbp_mmhg"] = visits_clean["sbp_mmhg"].where(visits_clean["sbp_mmhg"].between(70, 250)).astype("Int64")
visits_clean["weight_kg"] = pd.to_numeric(visits_clean["weight_kg"], errors="coerce")
visits_clean["visit_date"] = pd.to_datetime(visits_clean["visit_date"].where(visits_clean["visit_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")), format="%Y-%m-%d", errors="coerce")

visit
baseline     28533
week12       26529
BASELINE       256
Baseline       242
bl             235
baseline       228
BL             220
wk12           204
 week12        198
Week 12        195
WEEK12         186
Wk12           177
week 12        168
Name: count, dtype: int64[pyarrow]


In [18]:
display(visits_clean[visits_clean.duplicated(subset=["patient_id", "visit"], keep=False)])
visits_clean = visits_clean.sort_values("visit_time").drop_duplicates(subset=["patient_id", "visit"], keep="last")

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
5,P00004,baseline,2026-02-25,11:15,142,106.2,visits_north.csv
84,P00088,baseline,2026-03-10,09:30,153,68.2,visits_north.csv
85,P00088,baseline,2026-03-10,09:50,152,68.2,visits_north.csv
1208,P01341,week12,2026-05-26,15:10,121,72.3,visits_north.csv
...,...,...,...,...,...,...,...
55282,P28051,week12,2026-04-16,09:35,126,92.7,visits_south.csv
55472,P28232,baseline,2026-01-12,10:00,153,61.0,visits_south.csv
55473,P28232,baseline,2026-01-12,10:10,159,61.0,visits_south.csv
56248,P28953,baseline,2026-01-21,10:45,151,74.8,visits_south.csv


In [19]:
visits_clean = visits_clean.sort_values(["patient_id", "visit"]).reset_index(drop=True)[["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "source_file"]]

visits_summary = pd.DataFrame({
    "metric": ["visits", "baseline visits", "week12 visits", "visit_date missing", "visit_date min", "visit_date max", "sbp_mmhg missing", "sbp_mmhg min", "sbp_mmhg max", "weight_kg missing", "weight_kg mean", "visits from visits_north.csv", "visits from visits_south.csv"],
    "value": [len(visits_clean), (visits_clean["visit"] == "baseline").sum(), (visits_clean["visit"] == "week12").sum(), visits_clean["visit_date"].isna().sum(), visits_clean["visit_date"].min(), visits_clean["visit_date"].max(), visits_clean["sbp_mmhg"].isna().sum(), visits_clean["sbp_mmhg"].min(), visits_clean["sbp_mmhg"].max(), visits_clean["weight_kg"].isna().sum(), round(visits_clean["weight_kg"].mean(), 2), (visits_clean["source_file"] == "visits_north.csv").sum(), (visits_clean["source_file"] == "visits_south.csv").sum()]
})
visits_summary.to_csv(VISITS_SUMMARY_PATH, index=False)
visits_summary

,metric,value
0,visits,57181
1,baseline visits,29599
2,week12 visits,27582
3,visit_date missing,169
4,visit_date min,2026-01-05 00:00:00
5,visit_date max,2026-07-06 00:00:00
6,sbp_mmhg missing,456
7,sbp_mmhg min,70
8,sbp_mmhg max,250
9,weight_kg missing,828


### 3.3 Record the decisions

In [20]:
decision_table = pd.DataFrame([
    ("patients", "exact duplicate rows", "Keep the first copy of each exact duplicate.", "Identical registry rows repeat the same patient information."),
    ("patients", "gender spellings that are not a code", "Map spellings to F, M, or X; leave empty and unknown values missing.", "Different spellings represent the same categories, while empty and unknown values provide no recorded gender."),
    ("patients", "site codes needing normalization", "Uppercase site codes and remove all spaces and hyphens.", "Codes such as n-01 and N01 refer to the same site."),
    ("patients", "birth_year sentinel 9999", "Replace 9999 with missing and use Int64.", "9999 means the birth year was not recorded."),
    ("visits", "weight column named wt_kg in visits_south.csv", "Rename wt_kg to weight_kg before stacking.", "Both columns contain weight in kg and must align when stacked."),
    ("visits", "sbp sentinel 999 and readings outside 70 through 250", "Keep values from 70 through 250 inclusive; set all others to missing.", "999 means not measured, and readings outside 70 through 250 violate the specified range."),
    ("visits", "impossible and loosely written visit dates", "Keep only existing calendar dates written exactly as YYYY-MM-DD.", "Impossible dates and loosely written dates violate the required date format or calendar validity."),
    ("visits", "repeated patient_id and visit pairs", "Remove exact repeats, then keep the later reading for each patient and visit.", "Rechecks repeat a visit key, and the required rule retains the later measurement."),
    ("sites", "repeated site_code rows", "Keep only current records.", "Current and retired versions share site_code, but the analysis requires the current site name.")
], columns=["table", "issue", "decision", "reason"])
decision_table.to_csv(DECISION_LOG_PATH, index=False)
decision_table

,table,issue,decision,reason
0,patients,exact duplicate rows,Keep the first copy of each exact duplicate.,Identical registry rows repeat the same patien...
1,patients,gender spellings that are not a code,"Map spellings to F, M, or X; leave empty and u...",Different spellings represent the same categor...
2,patients,site codes needing normalization,Uppercase site codes and remove all spaces and...,Codes such as n-01 and N01 refer to the same s...
3,patients,birth_year sentinel 9999,Replace 9999 with missing and use Int64.,9999 means the birth year was not recorded.
4,visits,weight column named wt_kg in visits_south.csv,Rename wt_kg to weight_kg before stacking.,Both columns contain weight in kg and must ali...
5,visits,sbp sentinel 999 and readings outside 70 throu...,Keep values from 70 through 250 inclusive; set...,"999 means not measured, and readings outside 7..."
6,visits,impossible and loosely written visit dates,Keep only existing calendar dates written exac...,Impossible dates and loosely written dates vio...
7,visits,repeated patient_id and visit pairs,"Remove exact repeats, then keep the later read...","Rechecks repeat a visit key, and the required ..."
8,sites,repeated site_code rows,Keep only current records.,"Current and retired versions share site_code, ..."


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [ ]:
# TODO: melt raw["a1c_wide"], map the visit labels, drop the rows with no result,
# then save the per-visit summary to A1C_SUMMARY_PATH (README Task 4.1).
a1c_long = None

### 4.2 Check the keys before joining

In [ ]:
# TODO: steps 1 and 2, catch the MergeError, show the repeated site rows, and build sites_current.
sites = raw["sites"]
sites_current = None

In [ ]:
# TODO: step 3, outer merge visits_clean and a1c_long with indicator=True and show the _merge counts,
# then save the seven counts in README Task 4.2 to MERGE_AUDIT_PATH.

### 4.3 Pivot blood pressure to one row per patient

In [ ]:
# TODO: pivot visits_clean, reset the index, clear columns.name, add sbp_change, then save the
# SAMPLE_IDS rows to SBP_WIDE_SAMPLE_PATH (README Task 4.3).
sbp_wide = None

#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [ ]:
# TODO: select complete pairs, convert to a NumPy array, and display its shape and two per-visit means.

### 4.4 Join the analysis table

In [ ]:
# TODO: three left merges onto visits_clean, each with validate=, then the columns and dtypes
# in README Task 4.4. A MergeError means an earlier step left a problem: see HINTS.md.
analysis = None

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [ ]:
# TODO: collect named True/False checks in a Series (README Task 5.1), display it, and stop with
# assert checks.all(), checks[~checks] before the analysis extract is saved.
# A failed check means an earlier step left a problem: print checks[~checks] and see HINTS.md.

### 5.2 Save the analysis extract

In [ ]:
# TODO: keep the analysis rows whose patient_id is in SAMPLE_IDS, renumber them, and save them to
# ANALYSIS_SAMPLE_PATH with index=False (README Task 5.2).

### 5.3 Read the file back

In [ ]:
# TODO: read ANALYSIS_SAMPLE_PATH back with its dtypes and parse_dates=["visit_date"], and print
# whether it equals analysis_sample.